# Fixing Skewed GroupBy with Salting (Databricks / PySpark)

A single hot key in a `groupBy().agg()` can stall an entire Spark job. One executor gets stuck processing millions of rows for that key while the rest sit idle — the classic **data skew** bottleneck.

**Salting** breaks the hot key into `N` artificial sub-keys, spreads the work across `N` partitions, then merges the partial results. This notebook reproduces the problem with synthetic transaction data and walks through the fix step by step.

> This notebook uses pure PySpark and runs in Databricks, local Spark, or Colab (with `pyspark` installed).

In [ ]:
# Install dependencies if running in Colab
!pip install pyspark --quiet

## Create a Skewed Dataset

We simulate a transaction table where one `customer_id` (`"WHALE"`) accounts for ~50% of all rows. This is realistic — think a marketplace where one enterprise buyer generates orders of magnitude more transactions than typical users.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import random
import time
from datetime import datetime, timedelta

spark = SparkSession.builder.master("local[*]").appName("salting-demo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Generate skewed transaction data
random.seed(42)
n_normal = 200_000   # spread across 1000 customers
n_whale = 200_000    # ALL assigned to one customer

customers = [f"CUST_{i:04d}" for i in range(1000)]

rows = []
base_ts = datetime(2024, 1, 1)

# Normal customers: ~200 transactions each on average
for _ in range(n_normal):
    rows.append((
        random.choice(customers),
        round(random.uniform(5.0, 500.0), 2),
        base_ts + timedelta(seconds=random.randint(0, 31_536_000)),
    ))

# The whale: 200k transactions all under one key
for _ in range(n_whale):
    rows.append((
        "WHALE",
        round(random.uniform(5.0, 5000.0), 2),
        base_ts + timedelta(seconds=random.randint(0, 31_536_000)),
    ))

schema = StructType([
    StructField("customer_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("transaction_ts", TimestampType()),
])

df = spark.createDataFrame(rows, schema=schema)

# Show the skew
print("Row distribution:")
df.groupBy("customer_id").count().orderBy(F.desc("count")).show(5)

## The Problem: Naive GroupBy

A straightforward `groupBy("customer_id").agg(...)` sends all 200k `WHALE` rows to a single partition. In a real cluster this means one task takes 100x longer than the others, and the job's wall-clock time is bottlenecked by that single task.

Even in local mode you can see the time difference.

In [ ]:
start = time.time()

naive_result = (
    df
    .groupBy("customer_id")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.count("*").alias("txn_count"),
        F.min("transaction_ts").alias("first_txn"),
        F.max("transaction_ts").alias("last_txn"),
    )
)

# Force execution
naive_result.cache().count()
naive_time = time.time() - start

print(f"Naive groupBy: {naive_time:.2f}s")
naive_result.orderBy(F.desc("txn_count")).show(5, truncate=False)

## The Fix: Two-Phase Salted Aggregation

Salting works in two phases:

1. **Phase 1 (Scatter):** Append a random salt (`0..N-1`) to each `customer_id`, creating `N` sub-keys per customer. GroupBy on `(customer_id, salt)` and compute **partial** aggregates. The hot key's 200k rows are now split across `N` partitions.

2. **Phase 2 (Gather):** Drop the salt and groupBy on `customer_id` again, merging the `N` partial results into the final aggregates.

The key insight is that `SUM`, `COUNT`, `MIN`, and `MAX` are **decomposable** — you can split-then-merge them. `AVG` is not directly decomposable, so we compute `SUM` and `COUNT` in Phase 1 and derive the average in Phase 2.

In [ ]:
NUM_SALT_BUCKETS = 10

start = time.time()

# --- PHASE 1: Scatter ---
# Add a random salt column to spread the hot key across partitions
salted = df.withColumn("salt", (F.rand() * NUM_SALT_BUCKETS).cast("int"))

# Partial aggregation on (customer_id, salt)
# Note: we compute sum + count separately so we can derive avg in Phase 2
partial = (
    salted
    .groupBy("customer_id", "salt")
    .agg(
        F.sum("amount").alias("partial_sum"),
        F.count("*").alias("partial_count"),
        F.min("transaction_ts").alias("partial_min_ts"),
        F.max("transaction_ts").alias("partial_max_ts"),
    )
)

# --- PHASE 2: Gather ---
# Drop the salt and merge partial results into final aggregates
salted_result = (
    partial
    .groupBy("customer_id")
    .agg(
        F.sum("partial_sum").alias("total_amount"),
        F.sum("partial_count").alias("txn_count"),
        F.min("partial_min_ts").alias("first_txn"),
        F.max("partial_max_ts").alias("last_txn"),
    )
    # Derive average from total sum / total count
    .withColumn("avg_amount", F.col("total_amount") / F.col("txn_count"))
)

# Force execution
salted_result.cache().count()
salted_time = time.time() - start

print(f"Salted groupBy: {salted_time:.2f}s")
salted_result.orderBy(F.desc("txn_count")).show(5, truncate=False)

## Verify Correctness

Salting must produce the **exact same numbers** as the naive approach. Let's check.

In [ ]:
# Compare naive vs salted results for the WHALE customer
naive_whale = naive_result.filter(F.col("customer_id") == "WHALE").collect()[0]
salted_whale = salted_result.filter(F.col("customer_id") == "WHALE").collect()[0]

print("=== WHALE customer comparison ===")
print(f"{'Metric':<20} {'Naive':>20} {'Salted':>20}")
print("-" * 62)
for col in ["total_amount", "avg_amount", "txn_count", "first_txn", "last_txn"]:
    print(f"{col:<20} {str(naive_whale[col]):>20} {str(salted_whale[col]):>20}")

# Numeric validation
assert naive_whale["txn_count"] == salted_whale["txn_count"], "txn_count mismatch!"
assert abs(naive_whale["total_amount"] - salted_whale["total_amount"]) < 0.01, "total_amount mismatch!"
assert str(naive_whale["first_txn"]) == str(salted_whale["first_txn"]), "first_txn mismatch!"
assert str(naive_whale["last_txn"]) == str(salted_whale["last_txn"]), "last_txn mismatch!"

print("\nAll assertions passed -- salted result is identical to naive.")

## When to Use Salting

**Use salting when:**
- The Spark UI shows a few tasks taking 10-100x longer than the rest in a `groupBy` or `join` stage
- One or a small number of keys dominate the row count
- You're running on Databricks and AQE (Adaptive Query Execution) alone isn't enough — AQE handles join skew well but is less effective for aggregation skew

**Don't use salting when:**
- The skew is mild (< 5x difference between largest and median partition) — the overhead of the extra shuffle isn't worth it
- You can filter or pre-aggregate the hot key separately (simpler approach)
- You're using `MEDIAN`, `PERCENTILE`, or other non-decomposable aggregates — salting can't split these correctly

**Tuning `NUM_SALT_BUCKETS`:** Start with `10`. If the hot key has 100x more rows than the median key, try `50-100`. More buckets = more parallelism for the hot key, but also more intermediate groups in Phase 2.